<a href="https://colab.research.google.com/github/turkerbdonmez/XplainIt/blob/main/nb/Gemma4_(E2B)-Text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps git+https://github.com/huggingface/transformers.git
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
from unsloth import FastModel
import torch

# Ortalama ~700 token/örnek. 2048 neredeyse tüm örnekleri kapsar.
# Çok uzun cevaplar varsa 4096'ya çıkarabilirsin (VRAM artar).
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastModel.from_pretrained(
    model_name      = "unsloth/gemma-4-E4B-it",
    dtype           = None,
    max_seq_length  = MAX_SEQ_LENGTH,
    load_in_4bit    = True,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.1: Fast Gemma4 patching. Transformers: 5.5.0.dev0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # Vision kapalı — sadece metin
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r            = 32,   # 137K medikal örnek için yeterli kapasite
    lora_alpha   = 32,
    lora_dropout = 0,
    bias         = "none",
    random_state = 3407,
)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

In [ ]:
from datasets import load_dataset

DATA_DIR = "/content/drive/MyDrive/medikalcorpus"

dataset_train = load_dataset(
    "json",
    data_files = f"{DATA_DIR}/mega_train.jsonl",
    split      = "train",
)
dataset_valid = load_dataset(
    "json",
    data_files = f"{DATA_DIR}/mega_valid.jsonl",
    split      = "train",
)

print(f"Train: {len(dataset_train):,} örnek")
print(f"Valid: {len(dataset_valid):,} örnek")
print("\nİlk örnek (ham):")
print(dataset_train[0])

In [ ]:
def formatting_prompts_func(examples):
    """messages listesini Gemma-4 chat template'e çevirir."""
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize              = False,
            add_generation_prompt = False,
        ).removeprefix('<bos>')
        for convo in convos
    ]
    return {"text": texts}

dataset_train = dataset_train.map(formatting_prompts_func, batched=True)
dataset_valid = dataset_valid.map(formatting_prompts_func, batched=True)

print("Formatlanmış ilk örnek:")
print(dataset_train[0]["text"][:500])

In [ ]:
gpu_stats        = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory       = round(gpu_stats.total_memory       / 1024**3, 3)
print(f"GPU            : {gpu_stats.name}")
print(f"Toplam bellek  : {max_memory} GB")
print(f"Şu an ayrılan  : {start_gpu_memory} GB")

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset_train,
    eval_dataset  = dataset_valid,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LENGTH,
        packing                     = True,

        # Batch & gradient
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # Efektif batch = 8

        # Epoch & adım
        num_train_epochs            = 3,
        # max_steps                 = 500, # Hızlı test için bu satırı aç, üstteki epoch'u kapat

        # Learning rate
        learning_rate               = 1e-4,
        lr_scheduler_type           = "cosine",
        warmup_ratio                = 0.03,  # İlk ~1500 adım warmup

        # Regularizasyon
        weight_decay                = 0.01,

        # Logging & eval
        logging_steps               = 50,
        eval_strategy               = "steps",
        eval_steps                  = 500,

        # Checkpoint
        save_strategy               = "steps",
        save_steps                  = 1000,
        save_total_limit            = 3,    # En iyi 3 checkpoint'i tut
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",

        # Optimizasyon
        optim                       = "adamw_8bit",
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),

        # Çıktı
        output_dir                  = "/content/drive/MyDrive/medikalcorpus/checkpoints",
        seed                        = 3407,
        report_to                   = "none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part    = "<|turn>model\n",
)

# Maskeleme doğrulama — sadece assistant cevabı görünmeli
sample_ids     = trainer.train_dataset[0]["input_ids"]
sample_labels  = trainer.train_dataset[0]["labels"]
print("Input  :", tokenizer.decode(sample_ids)[:300])
print()
print("Labels :", tokenizer.decode(
    [tokenizer.pad_token_id if x == -100 else x for x in sample_labels]
).replace(tokenizer.pad_token, "_")[:300])

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory          = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_pct             = round(used_memory          / max_memory * 100, 2)
lora_pct             = round(used_memory_for_lora / max_memory * 100, 2)
runtime_sec          = trainer_stats.metrics['train_runtime']

print(f"Eğitim süresi        : {runtime_sec:.0f} sn  ({runtime_sec/3600:.2f} saat)")
print(f"Peak GPU bellek      : {used_memory} GB  (%{used_pct})")
print(f"LoRA için kullanılan : {used_memory_for_lora} GB  (%{lora_pct})")

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/medikalcorpus/gemma4-e4b-tusgpt-lora"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"LoRA adaptörleri kaydedildi: {SAVE_DIR}")